
# Modelado Predictivo con Aprendizaje Computacional
## Proyecto de Cambio Climático

Este notebook desarrolla la fase de modelado predictivo utilizando técnicas de aprendizaje supervisado sobre un dataset climático entre 2000 y 2023.



# 1. Importación de Librerías


In [6]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
#from sklearn.ensemble import RandomForestRegressorpio
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings('ignore')



# 2. Carga del Dataset


In [11]:

df = pd.read_csv(r'../data/raw/climate_change_dataset.csv')

df.head()


,Year,Country,Avg Temperature (°C),CO2 Emissions (Tons/Capita),Sea Level Rise (mm),Rainfall (mm),Population,Renewable Energy (%),Extreme Weather Events,Forest Area (%)
0,2006,UK,8.9,9.3,3.1,1441,530911230,20.4,14,59.8
1,2019,USA,31.0,4.8,4.2,2407,107364344,49.2,8,31.0
2,2014,France,33.9,2.8,2.2,1241,441101758,33.3,9,35.5
3,2010,Argentina,5.9,1.8,3.2,1892,1069669579,23.7,7,17.7
4,2007,Germany,26.9,5.6,2.4,1743,124079175,12.5,4,17.4



# 3. Comprensión del Dataset

El dataset contiene información relacionada con:
- Temperatura promedio.
- Emisiones de CO2.
- Nivel del mar.
- Precipitación.
- Energía renovable.
- Eventos climáticos extremos.
- Área forestal.

Cada fila representa un país en un año específico.


In [12]:

print(df.info())

print(df.describe())


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Year                         1000 non-null   int64  
 1   Country                      1000 non-null   str    
 2   Avg Temperature (°C)         1000 non-null   float64
 3   CO2 Emissions (Tons/Capita)  1000 non-null   float64
 4   Sea Level Rise (mm)          1000 non-null   float64
 5   Rainfall (mm)                1000 non-null   int64  
 6   Population                   1000 non-null   int64  
 7   Renewable Energy (%)         1000 non-null   float64
 8   Extreme Weather Events       1000 non-null   int64  
 9   Forest Area (%)              1000 non-null   float64
dtypes: float64(5), int64(4), str(1)
memory usage: 78.3 KB
None
              Year  Avg Temperature (°C)  CO2 Emissions (Tons/Capita)  \
count  1000.000000           1000.000000                  1000.000000   


# 4. Renombramiento de Variables


In [13]:

df.columns = [
    'year',
    'country',
    'avg_temperature_c',
    'co2_emissions_tons_capita',
    'sea_level_rise_mm',
    'rainfall_mm',
    'population',
    'renewable_energy_pct',
    'extreme_weather_events',
    'forest_area_pct'
]

df.head()


,year,country,avg_temperature_c,co2_emissions_tons_capita,sea_level_rise_mm,rainfall_mm,population,renewable_energy_pct,extreme_weather_events,forest_area_pct
0,2006,UK,8.9,9.3,3.1,1441,530911230,20.4,14,59.8
1,2019,USA,31.0,4.8,4.2,2407,107364344,49.2,8,31.0
2,2014,France,33.9,2.8,2.2,1241,441101758,33.3,9,35.5
3,2010,Argentina,5.9,1.8,3.2,1892,1069669579,23.7,7,17.7
4,2007,Germany,26.9,5.6,2.4,1743,124079175,12.5,4,17.4



# 5. Revisión de Valores Faltantes


In [14]:

df.isnull().sum()


year                         0
country                      0
avg_temperature_c            0
co2_emissions_tons_capita    0
sea_level_rise_mm            0
rainfall_mm                  0
population                   0
renewable_energy_pct         0
extreme_weather_events       0
forest_area_pct              0
dtype: int64


# 6. Variable Objetivo

Se selecciona como variable objetivo:

```python
avg_temperature_c
```

Esto debido a que representa uno de los principales indicadores del cambio climático.


In [15]:

X = df.drop(columns=['avg_temperature_c'])
y = df['avg_temperature_c']



# 7. Variables Numéricas y Categóricas


In [16]:

numeric_features = [
    'year',
    'co2_emissions_tons_capita',
    'sea_level_rise_mm',
    'rainfall_mm',
    'population',
    'renewable_energy_pct',
    'extreme_weather_events',
    'forest_area_pct'
]

categorical_features = ['country']



# 8. Transformaciones y Preprocesamiento


In [17]:

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])



# 9. División Train/Test


In [18]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print('Datos entrenamiento:', X_train.shape)
print('Datos prueba:', X_test.shape)


Datos entrenamiento: (800, 9)
Datos prueba: (200, 9)



# 10. Modelado Predictivo

Se evaluarán cuatro modelos:
Regresión Lineal

Se utilizó porque permite identificar relaciones lineales entre las variables climáticas y la temperatura, además de ser un modelo sencillo e interpretable para análisis predictivo.

Árbol de Decisión

Se aplicó debido a su capacidad para encontrar patrones y relaciones no lineales en los datos, facilitando además la interpretación de las decisiones del modelo.

Random Forest

Se utilizó porque mejora la precisión combinando múltiples árboles de decisión, reduciendo el sobreajuste y aumentando la estabilidad de las predicciones.

Gradient Boosting

Se seleccionó por su capacidad de optimizar el rendimiento predictivo mediante el aprendizaje secuencial de errores, logrando modelos más precisos.


## 10.1 Regresión Lineal


In [19]:

linear_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)



## 10.2 Árbol de Decisión


In [20]:

tree_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(
        max_depth=5,
        random_state=42
    ))
])

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)



## 10.3 Random Forest


In [21]:

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)



## 10.4 Gradient Boosting


In [22]:

gb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)

gb_predictions = gb_model.predict(X_test)
